# `stellarator_helias` — the SAND residual as one Warp kernel

Each cottax node is traced to a jaxpr and emitted as one `@wp.func`; the emitter walks the
graph in topological order and writes a single kernel `f(unknowns[], boundary[]) -> conditions[]`.
No resolver, no hand-written Warp. Below: build it, look at one node, check it against JAX,
time it.

In [1]:
import os, time
from pathlib import Path

REPO = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "functional_process").is_dir())
os.chdir(REPO)          # `cottax` is ~/jaxgraph/src/cottax, not functional_process/cottax

import numpy as np, jax
jax.config.update("jax_enable_x64", True)
import warp as wp
from functional_process.cottax.warp import measure as M
from functional_process.cottax.warp.jaxpr_backend import warp_init

prep = M.prepare("stellarator_helias")
print(f"{len(prep.usable)}/{prep.n_nodes} nodes   {len(prep.unknowns)} unknowns   "
      f"{len(prep.scalar_boundary)}+{len(prep.array_boundary)} boundary   "
      f"{len(prep.conditions)} conditions   {prep.n_eqns:,} jaxpr eqns")

[measure] stellarator_helias: 123/123 entries, 21/21 conditions, 14 unknowns, 308 scalar boundary, 11 array boundary; module 0.89 MB
123/123 nodes   14 unknowns   308+11 boundary   21 conditions   16,803 jaxpr eqns


### One node: jaxpr in, `@wp.func` out

In [2]:
from functional_process.cottax.warp.jaxpr_backend import node_values, emit_node
from functional_process.cottax.warp.assemble import _assemble
from functional_process.cottax import native as _native

drive, _, env = _assemble("stellarator_helias")
cold = _native.native_reference("tests/regression/input_files/stellarator_helias.IN.DAT").cold
g = drive.body.subgraph
npath = next(n for n in g.topological_order if n.path_str() == ".physics.fusion_rates")
defn = g[npath]
jx = jax.make_jaxpr(lambda *a: defn.fn(*a))(*node_values(cold, defn, mda_env=env))
print(f"jaxpr: {len(jx.jaxpr.eqns)} top-level eqns\n")
print("\n".join(str(jx).splitlines()[:6]), "\n  ...")

jaxpr: 407 top-level eqns

let _where = { lambda ; a:bool[201] b:f64[] c:f64[201]. let
    d:f64[] = convert_element_type[new_dtype=float64 weak_type=False] b
    e:f64[201] = broadcast_in_dim d
    f:f64[201] = select_n a c e
  in (f,) } in
{ lambda ; g:f64[201] h:f64[201] i:f64[201] j:f64[] k:f64[] l:f64[] m:f64[] n:f64[] 
  ...


In [3]:
src = next(e.source for e in prep.usable if e.node == ".physics.fusion_rates")
lines = src.splitlines()
print(f"emitted @wp.func: {len(lines)} lines\n")
print("\n".join(lines[:12]), "\n  ...\n", "\n".join(lines[-3:]))

emitted @wp.func: 455 lines

@wp.func
def n30_physics_fusion_rates(a0: wp.array(dtype=wp.float64), a1: vec201f, a2: vec201f, a3: wp.float64, a4: wp.float64, a5: wp.float64, a6: wp.float64, a7: wp.float64, a8: wp.float64, a9: wp.float64):
    t1 = (wp.float64(0.0083264) * a3)
    t2 = (wp.float64(1.02934) - t1)
    t3 = a3
    t4 = (t3 * t3)
    t5 = (wp.float64(0.00017631) * t4)
    t6 = (t2 + t5)
    t7 = a3
    t8 = (t7 * t7)
    t9 = (t7 * t8)
    t10 = (wp.float64(1.8201e-06) * t9) 
  ...
     t439 = (t438 + t420)
    t440 = (t405 + t417)
    return t424, t427, t430, t433, t436, t439, t210, t384, t394, t440, t17, t20, t22, t24, t26


### The kernel

In [4]:
import os
kern = [l for l in open(prep.module_path) if l.startswith("@wp.kernel") or l.startswith("def ")]
print("".join(kern[-2:]))
print(f"module: {prep.module_path}  ({os.path.getsize(prep.module_path)/1e6:.2f} MB)")

@wp.kernel
def stellarator_helias_jaxpr_subdag(x: wp.array2d(dtype=wp.float64), p: wp.array2d(dtype=wp.float64), v_costs_ucsc_buf: wp.array(dtype=wp.float64), v_costs_uchts_buf: wp.array(dtype=wp.float64), v_costs_ucturb_buf: wp.array(dtype=wp.float64), v_costs_cfind_buf: wp.array(dtype=wp.float64), v_costs_ucoam_buf: wp.array(dtype=wp.float64), v_costs_ucwst_buf: wp.array(dtype=wp.float64), v_physics_radius_plasma_profile_norm_buf: wp.array(dtype=wp.float64), v_impurity_radiation_temp_impurity_keV_array_buf: wp.array(dtype=wp.float64), v_impurity_radiation_pden_impurity_lz_nd_temp_array_buf: wp.array(dtype=wp.float64), v_impurity_radiation_impurity_arr_zav_buf: wp.array(dtype=wp.float64), v_impurity_radiation_m_impurity_amu_array_buf: wp.array(dtype=wp.float64), r: wp.array2d(dtype=wp.float64)):

module: functional_process/warp/_jaxpr_subdag_stellarator_helias.py  (0.89 MB)


### Agreement against JAX — same sub-DAG, same inputs

In [5]:
warp_init()
from functional_process.cottax.warp.jaxpr_harness import AGREEMENT_RTOL
k, _t, _cache = M.load_kernel(wp, prep, "cpu")
args, x, r = M._wp_inputs(wp, prep, 1, "cpu")
wp.launch(k, dim=1, inputs=args, device="cpu"); wp.synchronize()
r_wp = r.numpy()[0]

f = jax.jit(M.make_jax_single(prep))
x0, p0 = M._x0_p0(prep)
r_jx = np.asarray(f(x0, p0, *M.jax_array_args(prep)))

rel, worst, zabs, nnz, nz = M._rel(r_wp, r_jx)
print(f"worst relative difference: {worst:.3e}   (gate {AGREEMENT_RTOL:g})")
print(f"non-finite on BOTH engines: {int((~np.isfinite(r_jx)).sum())}/{len(r_jx)}  "
      f"(objf at this probe point)")

Warp 1.17.0 initialized:
   CUDA Toolkit 12.9, Driver 12.6
   Devices:
     "cpu"      : "x86_64"
     "cuda:0"   : "Quadro T1000 with Max-Q Design" (4 GiB, sm_75, mempool enabled)
   Kernel cache:
     /home/tbogaarts/.cache/warp/1.17.0


[measure] python-exec 1.10 s; 7 cached build(s) for this module; compiling for cpu


Module _jaxpr_subdag_stellarator_helias_0 23b44c8 load on device 'cpu' took 1.33 ms  (cached)
[measure] first launch on cpu: 0.20 s (warm -- cache hit); cache /home/tbogaarts/.cache/warp/1.17.0/wp__jaxpr_subdag_stellarator_helias_0_157b851_p59630_t135340742264640


worst relative difference: 6.183e-15   (gate 1e-12)
non-finite on BOTH engines: 1/21  (objf at this probe point)


### Timing — CPU, Warp against JAX

In [6]:
rows = []
for n in (1, 16, 256, 4096, 65536):
    args, _, _ = M._wp_inputs(wp, prep, n, "cpu")
    t_wp = M._time_launch(wp, k, args, n, "cpu", repeats=3 if n > 4096 else 20)

    xb = np.tile(np.asarray(x0), (n, 1)); pb = np.tile(np.asarray(p0), (n, 1))
    fb = jax.jit(jax.vmap(M.make_jax_single(prep), in_axes=(0, 0) + (None,) * len(prep.array_boundary)))
    arrs = M.jax_array_args(prep)
    fb(xb, pb, *arrs).block_until_ready()
    t0 = time.perf_counter()
    for _ in range(3): fb(xb, pb, *arrs).block_until_ready()
    t_jx = (time.perf_counter() - t0) / 3
    rows.append((n, t_wp, t_jx))

print(f"{'batch':>7}{'warp s':>12}{'us/pt':>10}{'jax s':>12}{'us/pt':>10}{'jax/warp':>10}")
for n, tw, tj in rows:
    print(f"{n:>7}{tw:>12.6f}{tw/n*1e6:>10.2f}{tj:>12.6f}{tj/n*1e6:>10.2f}{tj/tw:>10.2f}x")

  batch      warp s     us/pt       jax s     us/pt  jax/warp
      1    0.001573   1573.41    0.000599    598.96      0.38x
     16    0.024367   1522.92    0.003119    194.96      0.13x
    256    0.388590   1517.93    0.043812    171.14      0.11x
   4096    6.319827   1542.93    0.739586    180.56      0.12x
  65536   99.651636   1520.56   12.546493    191.44      0.13x


### Reading the table

`jax/warp` is JAX's time as a fraction of Warp's, so **0.13x means JAX is ~8x faster**.
Warp's per-point cost is flat (~1520 us) because it runs one thread per point serially;
JAX vectorises across the batch. Note also that statements are not work: the emitted
source is 16,803 equations against 111,718 before loops and fusion, but a loop re-executes
at runtime, so per-point cost went *up* (144 us/pt at 43 entries, 1520 at 123).

### GPU

Not shown, and the reason is measured: NVRTC is **OOM-killed compiling this kernel**, at
6.8 GB under a 6 GB cgroup cap and 11.4 GB uncapped -- even with the emitted source down to
0.85 MB from 7.37 MB. Source size is not what drives it; per-function optimisation cost is.
A reduced-coverage kernel (`M.prepare(..., max_entries=n)`) does build on CUDA, and on those
cuts the GPU crosses over CPU between batch 16 and 256 and reaches 14-37x at 65536 -- which
is the only place Warp wins, and it cannot yet be had at full coverage.

Reverse mode is off (`warp_init()` defaults `enable_backward=False`): Warp does not replay a
dynamic loop's body in the backward pass, so anything such a loop produces reads as zero in
later adjoints. Forward values are unaffected -- see `adjoint_probe.py`.